**Séance :** 01  
**Nom :** FANOU Précieux  
**Porteur # :** P2  
**Étape assignée :** 1.2 --> 1.3 --> 1.5 --> 1.10

# Étape 1.3 : Typer les colonnes

**Objectif** : donner à chaque colonne son bon type, avec une attention particulière aux 116 variables catégorielles (`cat1` à `cat116`).

| Famille | Nombre | Type visé |
|---|---|---|
| `id` | 1 | entier |
| `cat1` … `cat116` | 116 | `category` (modalités communes aux deux fichiers) |
| `cont1` … `cont14` | 14 | décimal (`float64`) |
| `loss` | 1 | décimal (`float64`), seulement dans `train` |

> On **type**, on ne **transforme** pas : pas de majuscules, pas de regroupement, pas d'encodage. Le choix de l'encodage des catégorielles est une décision du bloc 2 du projet.

> On **signale** aussi les colonnes à plus de 255 modalités : ce seuil de vigilance vient de la limite de scikit-learn sur les catégorielles natives (à vérifier dans sa documentation), il n'est pas dans l'énoncé.

## Bloc 1 : imports et chemins des fichiers

In [1]:
# ── Ce que fait ce bloc : importer les outils et trouver où sont les fichiers du sujet 04 ──
from pathlib import Path      # pour manipuler des chemins de fichiers proprement
import pandas as pd           # pour lire les CSV et manipuler les tableaux (DataFrame)

# Sur Kaggle, les données sont dans /kaggle/input/... ; en local, dans le dossier courant
RACINE_KAGGLE = Path("/kaggle/input")

if RACINE_KAGGLE.exists():
    # On cherche train.csv UNIQUEMENT dans le jeu du sujet 04,
    # pour ne pas lire le train.csv d'une autre compétition
    trouves = [p for p in RACINE_KAGGLE.rglob("train.csv") if "sujet-04" in str(p)]
    if not trouves:
        raise FileNotFoundError("train.csv du sujet 04 introuvable : ajoutez votre jeu (Add Input).")
    DOSSIER = trouves[0].parent     # le dossier qui contient train.csv
else:
    DOSSIER = Path(".")             # en local : le dossier où se trouve le notebook

CHEMIN_TRAIN = DOSSIER / "train.csv"            # sinistres AVEC le montant (loss)
CHEMIN_A_ESTIMER = DOSSIER / "a_estimer.csv"    # sinistres SANS le montant

# Les noms de colonnes de chaque famille, construits une seule fois
COLS_CAT = [f"cat{i}" for i in range(1, 117)]    # cat1 ... cat116
COLS_CONT = [f"cont{i}" for i in range(1, 15)]   # cont1 ... cont14

print("Dossier des données  :", DOSSIER)
print("train.csv existe     :", CHEMIN_TRAIN.exists())
print("a_estimer.csv existe :", CHEMIN_A_ESTIMER.exists())
print("Colonnes cat prévues :", len(COLS_CAT), "| colonnes cont prévues :", len(COLS_CONT))

# ── Résultat attendu ──
# Dossier des données  : /kaggle/input/datasets/prcieuxfanou/esgc-vak-sujet-04-fanou   (en local : .)
# train.csv existe     : True
# a_estimer.csv existe : True
# Colonnes cat prévues : 116 | colonnes cont prévues : 14

Dossier des données  : .
train.csv existe     : True
a_estimer.csv existe : True
Colonnes cat prévues : 116 | colonnes cont prévues : 14


## Bloc 2 : lire les deux CSV en forçant les catégorielles en texte

On dit à pandas de lire `cat1` à `cat116` comme du **texte**. Il ne doit rien deviner. On désactive aussi la détection automatique des valeurs manquantes : une modalité qui s'appellerait `NA` serait sinon transformée en valeur manquante sans prévenir.

In [2]:
# ── Ce que fait ce bloc : lire train et a_estimer avec les 116 cat forcées en texte ──
# dtype : on impose le type texte (str) aux 116 colonnes catégorielles
types_lecture = {c: str for c in COLS_CAT}

# keep_default_na=False : le texte "NA" reste du texte (pas une valeur manquante)
# na_values=[""]       : seule une case vraiment vide serait considérée comme manquante
train = pd.read_csv(CHEMIN_TRAIN, dtype=types_lecture, keep_default_na=False, na_values=[""])
a_estimer = pd.read_csv(CHEMIN_A_ESTIMER, dtype=types_lecture, keep_default_na=False, na_values=[""])

print("Forme de train     :", train.shape)
print("Forme de a_estimer :", a_estimer.shape)

# Mémoire utilisée AVANT conversion en category (pour comparer au bloc 4)
memoire_avant = train.memory_usage(deep=True).sum() / 1e6
print(f"Mémoire de train avant conversion : {memoire_avant:.1f} Mo")

# Type d'une colonne cat à ce stade : du texte, pas encore category
print("Type de cat1 dans train :", train["cat1"].dtype)

# ── Résultat attendu ──
# Forme de train     : (150654, 132)
# Forme de a_estimer : (37664, 131)
# Mémoire de train avant conversion : un nombre en Mo (plusieurs centaines de Mo)
# Type de cat1 dans train : str   (ou object selon la version de pandas)

Forme de train     : (150654, 132)
Forme de a_estimer : (37664, 131)
Mémoire de train avant conversion : 893.7 Mo
Type de cat1 dans train : str


## Bloc 3 : construire UNE liste de modalités par colonne, commune aux deux fichiers

C'est le piège principal. Si on typait chaque fichier séparément, une même lettre pourrait recevoir deux codes différents dans `train` et dans `a_estimer`, et le modèle serait faux sans aucune erreur. On prend donc l'**union** des modalités des deux fichiers, triée, et on l'applique aux deux.

In [3]:
# ── Ce que fait ce bloc : fabriquer, pour chaque cat, un type category commun aux deux fichiers ──
types_cat = {}   # dictionnaire : nom de colonne -> type category commun

for c in COLS_CAT:
    # union des modalités vues dans train ET dans a_estimer, puis tri pour un ordre stable
    modalites = sorted(set(train[c].unique()) | set(a_estimer[c].unique()))
    types_cat[c] = pd.CategoricalDtype(categories=modalites, ordered=False)

# Exemple pour une colonne : combien de modalités, et les 5 premières
exemple = types_cat["cat1"]
print("cat1 : nombre de modalités :", len(exemple.categories))
print("cat1 : premières modalités :", list(exemple.categories[:5]))
print("Nombre de types construits :", len(types_cat))

# ── Résultat attendu ──
# cat1 : nombre de modalités : 2   (cat1 est binaire d'après l'énoncé : minimum de 2 modalités)
# cat1 : premières modalités : ['A', 'B']
# Nombre de types construits : 116

cat1 : nombre de modalités : 2
cat1 : premières modalités : ['A', 'B']
Nombre de types construits : 116


## Bloc 4 : appliquer les types à toutes les colonnes

In [4]:
# ── Ce que fait ce bloc : convertir id, cat, cont et loss dans leur type final ──
# 1) Les 116 catégorielles : type category, AVEC la liste de modalités commune
for c in COLS_CAT:
    train[c] = train[c].astype(types_cat[c])
    a_estimer[c] = a_estimer[c].astype(types_cat[c])

# 2) id : entier
train["id"] = train["id"].astype("int64")
a_estimer["id"] = a_estimer["id"].astype("int64")

# 3) Les 14 continues : décimaux (float64)
for c in COLS_CONT:
    train[c] = train[c].astype("float64")
    a_estimer[c] = a_estimer[c].astype("float64")

# 4) loss : décimal, seulement dans train
train["loss"] = train["loss"].astype("float64")

# Mémoire APRES conversion : on compare à celle mesurée au bloc 2
memoire_apres = train.memory_usage(deep=True).sum() / 1e6
print(f"Mémoire de train avant : {memoire_avant:.1f} Mo")
print(f"Mémoire de train après : {memoire_apres:.1f} Mo")

# Nombre de colonnes par type dans train
print()
print("Types de train :")
display(train.dtypes.astype(str).value_counts().to_frame("nombre de colonnes"))

# ── Résultat attendu ──
# La mémoire baisse nettement après conversion en category.
# Types de train : category 116 | float64 15 (14 cont + loss) | int64 1 (id)

Mémoire de train avant : 893.7 Mo
Mémoire de train après : 37.2 Mo

Types de train :


,nombre de colonnes
category,116
float64,15
int64,1


## Bloc 5 : compter les modalités de chaque catégorielle

In [5]:
# ── Ce que fait ce bloc : mesurer le nombre de modalités de chacune des 116 cat ──
nb_modalites = pd.Series(
    {c: len(types_cat[c].categories) for c in COLS_CAT},
    name="nb_modalites"
)

print("Minimum de modalités :", nb_modalites.min())
print("Maximum de modalités :", nb_modalites.max())

# Répartition par tranche, pour voir la forme du problème
tranches = pd.cut(nb_modalites, bins=[0, 2, 5, 20, 100, 255, 400],
                  labels=["2", "3-5", "6-20", "21-100", "101-255", "256-400"])
print()
print("Nombre de colonnes par tranche de modalités :")
display(tranches.value_counts().sort_index().to_frame("nombre de colonnes"))

# Les 10 colonnes les plus riches en modalités
print()
print("Les 10 colonnes avec le plus de modalités :")
display(nb_modalites.sort_values(ascending=False).head(10).to_frame())

# ── Résultat attendu ──
# Minimum de modalités : 2
# Maximum de modalités : 326
# La plupart des colonnes sont binaires (2 modalités) ; quelques-unes en ont plus de 100.

Minimum de modalités : 2
Maximum de modalités : 326

Nombre de colonnes par tranche de modalités :


,nombre de colonnes
nb_modalites,
2,72
3-5,19
6-20,19
21-100,4
101-255,1
256-400,1



Les 10 colonnes avec le plus de modalités :


,nb_modalites
cat116,326
cat110,131
cat109,84
cat113,61
cat112,51
cat115,23
cat105,20
cat107,20
cat101,19
cat114,19


## Bloc 6 : signaler les colonnes à plus de 255 modalités

On ne choisit pas l'encodage ici : on repère seulement les colonnes qui poseront problème, et on vérifie que les codes entiers de pandas ne débordent pas.

In [6]:
# ── Ce que fait ce bloc : lister les colonnes au-dessus du seuil et vérifier leurs codes ──
SEUIL = 255   # seuil de vigilance (limite des catégorielles natives de scikit-learn)

grosses = nb_modalites[nb_modalites > SEUIL].index.tolist()
print(f"Colonnes avec plus de {SEUIL} modalités : {len(grosses)}")

if grosses:
    lignes = []
    for c in grosses:
        codes = train[c].cat.codes          # le code entier que pandas attribue à chaque modalité
        lignes.append({
            "colonne": c,
            "nb_modalites": int(nb_modalites[c]),
            "type_des_codes": str(codes.dtype),   # doit pouvoir contenir plus de 255 valeurs
            "code_max_observe": int(codes.max()),
        })
    display(pd.DataFrame(lignes))
else:
    print("Aucune colonne ne dépasse le seuil.")

# Les types de codes utilisés sur l'ensemble des 116 cat
types_codes = pd.Series({c: str(train[c].cat.codes.dtype) for c in COLS_CAT})
print()
print("Type des codes, sur les 116 colonnes :")
display(types_codes.value_counts().to_frame("nombre de colonnes"))

# ── Résultat attendu ──
# Au moins une colonne au-dessus de 255 modalités (l'énoncé annonce jusqu'à 326).
# Pour ces colonnes, le type des codes est int16 (pas int8, qui s'arrête à 127 modalités).

Colonnes avec plus de 255 modalités : 1


,colonne,nb_modalites,type_des_codes,code_max_observe
0,cat116,326,int16,325



Type des codes, sur les 116 colonnes :


,nombre de colonnes
int8,114
int16,2


## Bloc 7 : modalités présentes dans un seul des deux fichiers

Une modalité qui n'existe que dans `a_estimer` n'a jamais été vue à l'entraînement : le modèle ne saura pas quoi en faire. On la détecte et on la compte, sans la corriger.

In [7]:
# ── Ce que fait ce bloc : repérer les modalités vues dans un seul fichier ──
lignes_nouvelles = []

for c in COLS_CAT:
    vues_train = set(train[c].unique())
    vues_a_estimer = set(a_estimer[c].unique())

    seulement_a_estimer = vues_a_estimer - vues_train   # jamais vues à l'entraînement
    seulement_train = vues_train - vues_a_estimer       # absentes de a_estimer

    if seulement_a_estimer or seulement_train:
        lignes_nouvelles.append({
            "colonne": c,
            "seulement_dans_a_estimer": len(seulement_a_estimer),
            "lignes_a_estimer_concernees": int(a_estimer[c].isin(seulement_a_estimer).sum()),
            "seulement_dans_train": len(seulement_train),
        })

nouvelles = pd.DataFrame(lignes_nouvelles)

print("Colonnes avec au moins une modalité vue dans un seul fichier :", len(nouvelles))
if len(nouvelles) > 0:
    print("Modalités jamais vues à l'entraînement (total) :", int(nouvelles["seulement_dans_a_estimer"].sum()))
    print("Lignes de a_estimer concernées (cumul par colonne) :", int(nouvelles["lignes_a_estimer_concernees"].sum()))
    display(nouvelles.sort_values("lignes_a_estimer_concernees", ascending=False).head(10))

# ── Résultat attendu ──
# Un nombre de colonnes concernées (peut être 0 ou plus). C'est une MESURE : on ne la suppose pas.
# Si le total est supérieur à 0, il faudra décider au bloc 2 comment traiter ces modalités.

Colonnes avec au moins une modalité vue dans un seul fichier : 18
Modalités jamais vues à l'entraînement (total) : 16
Lignes de a_estimer concernées (cumul par colonne) : 17


,colonne,seulement_dans_a_estimer,lignes_a_estimer_concernees,seulement_dans_train
17,cat116,9,10,68
11,cat109,4,4,11
4,cat101,1,1,2
1,cat89,1,1,2
12,cat110,1,1,17
0,cat75,0,0,1
6,cat103,0,0,1
2,cat90,0,0,2
3,cat92,0,0,1
5,cat102,0,0,2


## Bloc 8 : tableau récapitulatif de l'étape 1.3

In [8]:
# ── Ce que fait ce bloc : regrouper tous les contrôles du typage dans un seul tableau ──
def nb_colonnes(df, colonnes, test):
    """Compte combien de colonnes de la liste passent le test de type."""
    return sum(1 for c in colonnes if test(df[c]))

# Pour chaque catégorielle : aucun code négatif (-1 voudrait dire "valeur hors liste de modalités")
nb_codes_valides = sum(
    1 for c in COLS_CAT
    if train[c].cat.codes.min() >= 0 and a_estimer[c].cat.codes.min() >= 0
)

# Pour chaque catégorielle : les modalités sont-elles identiques dans les deux fichiers ?
nb_modalites_identiques = sum(
    1 for c in COLS_CAT
    if train[c].cat.categories.equals(a_estimer[c].cat.categories)
)

controles = [
    ("Colonnes cat présentes dans train",        116, sum(c in train.columns for c in COLS_CAT)),
    ("Colonnes cont présentes dans train",       14,  sum(c in train.columns for c in COLS_CONT)),
    ("Valeurs manquantes dans train",            0,   int(train.isna().sum().sum())),
    ("Valeurs manquantes dans a_estimer",        0,   int(a_estimer.isna().sum().sum())),
    ("cat en type category (train)",             116, nb_colonnes(train, COLS_CAT, lambda s: isinstance(s.dtype, pd.CategoricalDtype))),
    ("cat en type category (a_estimer)",         116, nb_colonnes(a_estimer, COLS_CAT, lambda s: isinstance(s.dtype, pd.CategoricalDtype))),
    ("cat avec modalités identiques (2 fichiers)", 116, nb_modalites_identiques),
    ("cat sans code négatif (aucune valeur hors liste)", 116, nb_codes_valides),
    ("cont en float64 (train)",                  14,  nb_colonnes(train, COLS_CONT, lambda s: str(s.dtype) == "float64")),
    ("cont en float64 (a_estimer)",              14,  nb_colonnes(a_estimer, COLS_CONT, lambda s: str(s.dtype) == "float64")),
    ("id en entier (train)",                     1,   int(pd.api.types.is_integer_dtype(train["id"]))),
    ("loss en float64 (train)",                  1,   int(str(train["loss"].dtype) == "float64")),
    ("Minimum de modalités d'une cat",           2,   int(nb_modalites.min())),
    ("Maximum de modalités d'une cat",           326, int(nb_modalites.max())),
]

recap = pd.DataFrame(controles, columns=["Contrôle", "Attendu", "Obtenu"])
recap["Verdict"] = recap.apply(
    lambda ligne: "OK" if ligne["Obtenu"] == ligne["Attendu"] else "ATTENTION", axis=1
)
display(recap)

# Information (pas un contrôle) : colonnes au-dessus du seuil, et modalités jamais vues
print(f"À signaler : {len(grosses)} colonne(s) au-dessus de {SEUIL} modalités.")
print(f"À signaler : {len(nouvelles)} colonne(s) avec des modalités vues dans un seul fichier.")

if (recap["Verdict"] == "OK").all():
    print("Étape 1.3 validée : tous les types sont conformes.")
else:
    print("ATTENTION : au moins un contrôle ne passe pas. Regarder la colonne Verdict.")

# ── Résultat attendu ──
# Un tableau de 14 lignes, toutes en OK, puis :
# Étape 1.3 validée : tous les types sont conformes.
# (Si le minimum ou le maximum de modalités diffère de 2 et 326, c'est une information à noter dans le journal.)

,Contrôle,Attendu,Obtenu,Verdict
0,Colonnes cat présentes dans train,116,116,OK
1,Colonnes cont présentes dans train,14,14,OK
2,Valeurs manquantes dans train,0,0,OK
3,Valeurs manquantes dans a_estimer,0,0,OK
4,cat en type category (train),116,116,OK
5,cat en type category (a_estimer),116,116,OK
6,cat avec modalités identiques (2 fichiers),116,116,OK
7,cat sans code négatif (aucune valeur hors liste),116,116,OK
8,cont en float64 (train),14,14,OK
9,cont en float64 (a_estimer),14,14,OK


À signaler : 1 colonne(s) au-dessus de 255 modalités.
À signaler : 18 colonne(s) avec des modalités vues dans un seul fichier.
Étape 1.3 validée : tous les types sont conformes.
